[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Pruebas para dos grupos.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se compararán dos grupos con las pruebas *t* de Student y de Welch, la prueba de Mann-Whitney y las pruebas por permutación, según la forma de los datos.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [2]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

((342, 7), (891, 15), (244, 8))

---
## 1. Dos grupos.

| Prueba | Compara | Supuestos | En *SciPy* |
|---|---|---|---|
| *t* de Student | Medias | Independencia, normalidad aproximada (o *n* grande), **varianzas iguales** | `ttest_ind(a, b)` |
| *t* de Welch | Medias | Independencia, normalidad aproximada; **no** exige varianzas iguales | `ttest_ind(a, b, equal_var=False)` |
| Mann-Whitney | Si un grupo tiende a tener valores mayores (rangos) | Independencia; robusta a atípicos y asimetría, pero con dispersiones o formas muy distintas puede dar un *p* pequeño aunque el centro no cambie | `mannwhitneyu(a, b)` |
| Permutación | Cualquier estadístico (diferencia de medias, de medianas…) | Independencia e intercambiabilidad bajo H0 | `permutation_test((a, b), estadistico)` |

**Decisión práctica:** usar **Welch por defecto** para medias (ojo: en *SciPy*, `ttest_ind` usa por defecto la versión de Student, así que Welch hay que pedirlo con `equal_var=False`); Mann-Whitney o permutación de medianas si hay asimetría fuerte o atípicos con *n* pequeño. La **independencia** no se arregla con ninguna prueba: mediciones repetidas del mismo individuo requieren pruebas pareadas.

**¿Qué hacer si falla un supuesto?**

| Si ocurre esto… | Entonces… |
|---|---|
| Las varianzas de los grupos son distintas | Usar Welch |
| Los datos son muy asimétricos o tienen atípicos y hay pocos datos | Usar Mann-Whitney o una permutación |
| Hay mediciones repetidas del mismo individuo | Usar una prueba para datos **pareados** |
| Los datos no son independientes (por ejemplo, vienen de la misma familia o de la misma mesa) | Ninguna prueba lo arregla: hay que cambiar el diseño |

**Concepto: Student contra Welch.** Las dos calculan lo mismo, la diferencia de medias medida en errores estándar, pero estiman el error estándar de forma distinta:
* **Student** junta las variaciones de los dos grupos en una sola, como si los dos variaran igual. Si el grupo grande varía poco y el pequeño mucho, esa variación combinada queda dominada por el grupo grande y **subestima** el error del grupo pequeño: la prueba se vuelve demasiado confiada.
* **Welch** usa la variación de cada grupo por separado y hace un ajuste para compensarlo. Por eso funciona bien tanto si los grupos varían igual como si no.

**Ejemplo cotidiano:** comparar el tiempo de traslado de 100 personas que van en metro (todas tardan parecido) con el de 10 que van en coche (unas tardan 20 minutos y otras 90). Juntar las dos variaciones haría creer que los del coche son tan predecibles como los del metro.

**Consecuencia: ¿Student o Welch?** Se simularán 5,000 comparaciones con H0 cierta (medias iguales), pero con un grupo pequeño y muy variable (*n* = 10, con desviación estándar de 3) y otro grande y poco variable (*n* = 50, con desviación estándar de 1). Con α = 0.05, la prueba debería equivocarse 5% de las veces.

* La siguiente celda calculará la tasa real de error tipo I de Student y de Welch en ese escenario.

In [3]:
pequeno_variable = rng.normal(0, 3, size=(5000, 10))
grande_estable = rng.normal(0, 1, size=(5000, 50))
{"Student": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1).pvalue < 0.05).mean()),
 "Welch": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1, equal_var=False).pvalue < 0.05).mean())}

{'Student': 0.2802, 'Welch': 0.0492}

Student rechaza H0 en torno a 28% de las veces (¡más de cinco veces α!), mientras que Welch se mantiene en ≈ 5%. Cuando las varianzas son iguales, Welch pierde muy poca potencia; por eso es la opción recomendada por defecto (en *SciPy* hay que pedirla con `equal_var=False`).

**Ejemplo:** masa de machos y hembras Adelie. Se revisan los supuestos, se aplican las pruebas y se calcula el tamaño del efecto.

**¿Por qué este orden?** En el flujo del EDA, una comparación de dos grupos se hace en cuatro pasos: (1) describir cada grupo (*n*, centro, dispersión; `13_agregacion`), (2) revisar supuestos, (3) aplicar la prueba y (4) cuantificar la magnitud con su IC. Primero se define la función del tamaño del efecto para usarla al final.

* La siguiente celda definirá `d_cohen`, la diferencia de medias dividida entre la desviación estándar combinada.

In [4]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)

* La siguiente celda separará las masas de machos y hembras Adelie y revisará normalidad (Shapiro-Wilk) e igualdad de varianzas (Levene).

In [5]:
adelie = pinguinos[pinguinos["species"] == "Adelie"]
machos_adelie = adelie.loc[adelie["sex"] == "Male", "body_mass_g"].to_numpy()
hembras_adelie = adelie.loc[adelie["sex"] == "Female", "body_mass_g"].to_numpy()
{"n": (len(machos_adelie), len(hembras_adelie)),
 "Shapiro machos p": round(float(stats.shapiro(machos_adelie).pvalue), 3),
 "Shapiro hembras p": round(float(stats.shapiro(hembras_adelie).pvalue), 3),
 "Levene p": round(float(stats.levene(machos_adelie, hembras_adelie).pvalue), 3)}

{'n': (73, 73),
 'Shapiro machos p': 0.416,
 'Shapiro hembras p': 0.199,
 'Levene p': 0.051}

**Interpretación:** con 73 machos y 73 hembras, Shapiro-Wilk no rechaza la normalidad en ningún grupo (*p* = 0.42 y 0.20) y Levene queda en el límite (*p* = 0.051; es la versión con la mediana, que usa *SciPy*; la versión clásica, con el promedio, da 0.035). Como se usará Welch, la igualdad de varianzas no hace falta; la normalidad aproximada y *n* > 30 respaldan la prueba *t*.

> 💡 **Qué preguntan estas dos pruebas.** Shapiro-Wilk pregunta si los datos se parecen a una campana, y Levene pregunta si dos grupos varían parecido; un valor *p* pequeño dice que «no». Con muy pocos datos casi nunca rechazan, y con muchísimos rechazan por detalles irrelevantes, así que siempre se miran también las gráficas.

**Nota:** las pruebas de normalidad tienen poca potencia con *n* pequeño y rechazan por detalles irrelevantes con *n* grande. Conviene complementarlas con un histograma o un gráfico Q-Q, y recordar que con *n* ≥ 30 por grupo la prueba *t* es robusta a desviaciones moderadas (pero no a datos muy asimétricos, como la tarifa del Titanic en `16_distribucion_muestral.ipynb`).

Un **gráfico Q-Q** coloca tus datos ordenados contra los valores que tendrían si vinieran de una campana. Si los puntos siguen la diagonal, la forma es parecida a la de una campana; si se curvan en los extremos, hay colas pesadas o asimetría.

**Concepto: tres maneras de obtener la distribución bajo H0.**
* **Welch:** con la fórmula de la *t*, que supone normalidad aproximada de las medias (TCL).
* **Mann-Whitney:** ordena todos los valores juntos y compara las **posiciones** (rangos) de cada grupo: si un grupo tiende a tener valores mayores, sus posiciones se acumulan arriba. Si H0 es cierta, las posiciones se reparten al azar entre los dos grupos. No usa los valores, así que es robusta a atípicos (aunque con dispersiones o formas muy distintas puede dar un *p* pequeño sin que cambie el centro). Ojo: compara posiciones (si un grupo tiende a tener valores mayores) y solo equivale a comparar medianas si las dos distribuciones tienen forma parecida; para comparar medianas se usa la permutación de medianas.
* **Permutación:** si H0 es cierta, las etiquetas "macho" y "hembra" son intercambiables. Se revuelven miles de veces, se recalcula la diferencia de medias en cada revuelta y el valor *p* es la fracción de revueltas con una diferencia al menos tan extrema como la observada. No supone una forma particular de la distribución, pero sí independencia e intercambiabilidad bajo H0; la permutación de promedios sigue siendo sensible a valores extremos y a dispersiones muy distintas entre los grupos.

* La siguiente celda aplicará Welch, Mann-Whitney y una prueba por permutación de la diferencia de medias (9,999 permutaciones, `random_state=401`).

In [6]:
welch = stats.ttest_ind(machos_adelie, hembras_adelie, equal_var=False)
mann_whitney = stats.mannwhitneyu(machos_adelie, hembras_adelie)
permutacion = stats.permutation_test((machos_adelie, hembras_adelie),
                                     lambda a, b, axis=-1: np.mean(a, axis=axis) - np.mean(b, axis=axis),
                                     n_resamples=9999, random_state=401)
pd.DataFrame({
    "estadístico": [welch.statistic, mann_whitney.statistic, permutacion.statistic],
    "valor p": [welch.pvalue, mann_whitney.pvalue, permutacion.pvalue],
}, index=["Welch", "Mann-Whitney", "permutación"])

,estadístico,valor p
Welch,13.126286,6.402320e-26
Mann-Whitney,5018.500000,3.091127e-20
permutación,674.657534,2.000000e-04


* La siguiente celda calculará la diferencia de medias con su IC del 95% (Welch) y la d de Cohen.

In [7]:
ic_welch = welch.confidence_interval(confidence_level=0.95)
(round(float(machos_adelie.mean() - hembras_adelie.mean()), 1), np.round([ic_welch.low, ic_welch.high], 1),
 round(float(d_cohen(machos_adelie, hembras_adelie)), 2))

(674.7, array([573. , 776.3]), 2.17)

Las tres pruebas coinciden: *p* minúsculo. El valor *p* de la permutación (0.0002) es el mínimo que entrega *SciPy* con 9,999 permutaciones en una prueba bilateral (otra convención daría 0.0001): ninguna permutación igualó la diferencia observada, así que el valor real es todavía menor. Lo informativo es la magnitud: los machos pesan unos 675 g más (IC 95%: 573 a 776 g), una d de 2.17, un efecto **muy grande**.

| d de Cohen (orientativo) | 0.2 | 0.5 | 0.8 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

📖 **Para profundizar (opcional).** *Las fórmulas de la sección 1.* Ambas pruebas usan $t = (\bar{x}_A - \bar{x}_B) / EE$, con

$$EE_{\text{Student}} = s_p\sqrt{\frac{1}{n_A} + \frac{1}{n_B}} \qquad EE_{\text{Welch}} = \sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}$$

donde $s_p$ es la desviación estándar combinada (ver `19_logica_del_contraste.ipynb`). Welch, además, ajusta los grados de libertad. El estadístico de Mann-Whitney es $U = R_A - n_A(n_A + 1)/2$, donde $R_A$ es la suma de los rangos del grupo A.

**💡 Ideas clave de la sección 1:**
* Welch por defecto; Mann-Whitney o permutación con asimetría o atípicos.
* Los supuestos se revisan con pruebas **y** gráficas; la independencia no se puede "probar", se razona sobre el diseño.
* Se reporta la diferencia con su IC y la d de Cohen, no solo el valor *p*.

### Ejercicio 1 · "No significativo" no es "igual".

**🎯 Objetivo:** aplicar Welch con su IC y su tamaño del efecto a un caso con *p* grande. **Qué se espera aprender:** a leer el IC para decir qué diferencias siguen siendo compatibles con los datos cuando no se rechaza H0.

Compara con Welch la masa de los Chinstrap y la de los Adelie. Reporta la diferencia de medias, su IC del 95%, el valor *p* y la d de Cohen. ¿Se puede afirmar que pesan lo mismo?

* La siguiente celda muestra una solución posible.

In [8]:
masa_chinstrap = pinguinos.loc[pinguinos["species"] == "Chinstrap", "body_mass_g"].to_numpy()
masa_adelie = adelie["body_mass_g"].dropna().to_numpy()
prueba = stats.ttest_ind(masa_chinstrap, masa_adelie, equal_var=False)
ic = prueba.confidence_interval()
{"diferencia": round(float(masa_chinstrap.mean() - masa_adelie.mean()), 1),
 "IC 95%": (round(float(ic.low), 1), round(float(ic.high), 1)),
 "p": round(float(prueba.pvalue), 3),
 "d": round(float(d_cohen(masa_chinstrap, masa_adelie)), 2)}

{'diferencia': 32.4, 'IC 95%': (-85.5, 150.4), 'p': 0.588, 'd': 0.07}

**✅ Respuesta sugerida:**

La diferencia es de 32 g (IC 95%: −86 a 150 g; *p* = 0.59; d = 0.07). No se rechaza H0, pero eso **no** demuestra que pesen lo mismo: el IC va de −86 g a +150 g (de −0.20 a +0.34 desviaciones estándar, con una desviación estándar de unos 440 g). No es simétrico: los datos son compatibles con que los Chinstrap pesen hasta unos 86 g menos o hasta unos 150 g más que los Adelie. Lo que sí se puede decir es que estos datos descartan diferencias mucho mayores que esas (150 g es cerca del 4% de la masa promedio de un Adelie).

### Ejercicio 2 · Propina porcentual de fumadores y no fumadores.

**🎯 Objetivo:** elegir pruebas robustas para una variable asimétrica con atípicos. **Qué se espera aprender:** a comparar posiciones con Mann-Whitney y **medianas** con una permutación, y a reportar los centros de cada grupo.

`tip_pct` es asimétrica y tiene atípicos (las medidas robustas se vieron en los *notebooks* `10` a `12`). Compara fumadores (`smoker == "Yes"`) y no fumadores con Mann-Whitney y con una prueba por permutación de la **diferencia de medianas** (`random_state=401`). Reporta las medianas de cada grupo.

* La siguiente celda muestra una solución posible.

In [9]:
fuman = propinas.loc[propinas["smoker"] == "Yes", "tip_pct"].to_numpy()
no_fuman = propinas.loc[propinas["smoker"] == "No", "tip_pct"].to_numpy()
perm_medianas = stats.permutation_test((fuman, no_fuman),
                                       lambda a, b, axis=-1: np.median(a, axis=axis) - np.median(b, axis=axis),
                                       n_resamples=9999, random_state=401)
{"mediana fuman": round(float(np.median(fuman)), 2), "mediana no fuman": round(float(np.median(no_fuman)), 2),
 "Mann-Whitney p": round(float(stats.mannwhitneyu(fuman, no_fuman).pvalue), 3),
 "permutación p": round(float(perm_medianas.pvalue), 3)}

{'mediana fuman': 15.38,
 'mediana no fuman': 15.56,
 'Mann-Whitney p': 0.561,
 'permutación p': 0.819}

**✅ Respuesta sugerida:**

Las medianas son casi iguales (15.38% en fumadores y 15.56% en no fumadores) y ninguna prueba encuentra diferencia (Mann-Whitney *p* = 0.56; permutación de medianas *p* = 0.82). En estos datos, fumar no se asocia con la propina porcentual típica. Ojo: la dispersión sí podría diferir (los fumadores tienen la propina extrema de 71%); comparar centros no lo detecta. Además, los datos son observacionales: no permiten decir que fumar cambie la propina.

### Reto 1 · Verifica la potencia por simulación.

**🎯 Objetivo:** comprobar por simulación un cálculo de potencia. **Qué se espera aprender:** qué significa en la práctica "80% de potencia" y cuánto cae al reducir *n*.

`TTestIndPower` dijo que con d = 0.5 hacen falta unos 64 casos por grupo para 80% de potencia. Compruébalo: simula 5,000 pares de muestras de tamaño 64 (una con media 0 y otra con media 0.5, ambas con desviación estándar 1; usa `generador = np.random.default_rng(401)`) y calcula la fracción de pruebas de Welch con *p* < 0.05. Repite con *n* = 20.

* La siguiente celda muestra una solución posible.

In [10]:
generador = np.random.default_rng(401)
potencia_simulada = {}
for n in [20, 64]:
    control = generador.normal(0, 1, size=(5000, n))
    tratamiento = generador.normal(0.5, 1, size=(5000, n))
    potencia_simulada[n] = float((stats.ttest_ind(tratamiento, control, axis=1, equal_var=False).pvalue < 0.05).mean())
potencia_simulada

{20: 0.3196, 64: 0.7924}

**✅ Respuesta sugerida:**

Con *n* = 64 la potencia simulada es ≈ 0.79, prácticamente el 80% que predice `TTestIndPower`; con *n* = 20 cae a ≈ 0.32: un efecto mediano real pasaría inadvertido dos de cada tres veces.

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios pendientes de este *notebook* y, si es posible, un reto.

**Siguiente *notebook*:** `21_proporciones_y_mas_de_dos_grupos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>